In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch.nn import BCEWithLogitsLoss, Linear, ModuleDict, LeakyReLU
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, accuracy_score, precision_recall_curve
import time
import psutil
import os
from torch_geometric.utils import degree
import scipy.stats as st

# --------------------------- Reproducibility & Device ---------------------------
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --------------------------- Load & Clean Data ---------------------------
nodes = pd.read_csv("nodes.tsv", sep="\t")
edges = pd.read_csv("edges.tsv", sep="\t")

for col in ["id", "kind"]:
    nodes[col] = nodes[col].astype(str).str.strip()

for col in ["source", "target", "metaedge"]:
    edges[col] = edges[col].astype(str).str.strip()

nodes = nodes.dropna(subset=["id", "kind"]).drop_duplicates(subset="id").reset_index(drop=True)
edges = edges.dropna(subset=["source", "target", "metaedge"]).copy()
edges = edges[edges["source"] != edges["target"]].copy()

tasks = ["CaD", "CrC", "DrD", "DaG"]
edges = edges[edges["metaedge"].isin(tasks)].copy()

# --------------------------- Encoding & Mapping ---------------------------
le_kind = LabelEncoder()
nodes["kind_encoded"] = le_kind.fit_transform(nodes["kind"])

le_metaedge = LabelEncoder()
edges["metaedge_encoded"] = le_metaedge.fit_transform(edges["metaedge"])

active_nodes = set(edges["source"]).union(set(edges["target"]))
nodes = nodes[nodes["id"].isin(active_nodes)].reset_index(drop=True)

node_id_map = {nid: idx for idx, nid in enumerate(nodes["id"])}
num_nodes = len(nodes)
num_kind_classes = len(le_kind.classes_)
num_edge_types = len(le_metaedge.classes_)

# --------------------------- Kind Mappings ---------------------------
compound_kind = le_kind.transform(["Compound"])[0]
disease_kind = le_kind.transform(["Disease"])[0]
gene_kind = le_kind.transform(["Gene"])[0]

task_to_kinds = {"CrC": (compound_kind, compound_kind), "DrD": (disease_kind, disease_kind), "CaD": (compound_kind, disease_kind), "DaG": (disease_kind, gene_kind)}

kind_to_nodes = {k: [] for k in range(num_kind_classes)}
for idx, kind in enumerate(nodes["kind_encoded"]):
    kind_to_nodes[kind].append(idx)

task_pos_weights = {"CaD": 2.0, "CrC": 0.5, "DrD": 2.0, "DaG": 0.5}

# --------------------------- Stratified Split by Node Kind Pairs ---------------------------
def get_edge_kind_pair(src_idx, tgt_idx, nodes_df):
    src_kind = nodes_df.iloc[src_idx]["kind_encoded"]
    tgt_kind = nodes_df.iloc[tgt_idx]["kind_encoded"]
    return (int(src_kind), int(tgt_kind))

def split_data_stratified(edge_index, edge_attr, nodes_df, val_ratio=0.2, test_ratio=0.2, seed=42):
    num_edges = edge_index.size(1)
    indices = torch.randperm(num_edges, generator=torch.Generator().manual_seed(seed))
    kind_pairs = []
    for i in range(num_edges):
        src = edge_index[0, i].item()
        tgt = edge_index[1, i].item()
        kind_pairs.append(get_edge_kind_pair(src, tgt, nodes_df))

    unique_pairs = list(set(kind_pairs))
    pair_to_indices = {pair: [] for pair in unique_pairs}
    for idx, pair in enumerate(kind_pairs):
        pair_to_indices[pair].append(indices[idx].item())

    train_idx_list = []
    val_idx_list = []
    test_idx_list = []

    for pair, pair_indices in pair_to_indices.items():
        n = len(pair_indices)
        if n == 0:
            continue
        train_size = int(n * (1 - val_ratio - test_ratio))
        val_size = int(n * val_ratio)
        np.random.seed(seed)
        np.random.shuffle(pair_indices)

        train_idx_list.extend(pair_indices[:train_size])
        val_idx_list.extend(pair_indices[train_size:train_size + val_size])
        test_idx_list.extend(pair_indices[train_size + val_size:])

    train_idx = torch.tensor(train_idx_list, dtype=torch.long)
    val_idx = torch.tensor(val_idx_list, dtype=torch.long)
    test_idx = torch.tensor(test_idx_list, dtype=torch.long)

    return {'train': {
            'edge_index': edge_index[:, train_idx],
            'edge_attr': edge_attr[train_idx]},
        'val': {
            'edge_index': edge_index[:, val_idx],
            'edge_attr': edge_attr[val_idx]},
        'test': {
            'edge_index': edge_index[:, test_idx],
            'edge_attr': edge_attr[test_idx]}}

edge_list = []
attr_list = []
for _, row in edges.iterrows():
    if row["source"] in node_id_map and row["target"] in node_id_map:
        edge_list.append([node_id_map[row["source"]], node_id_map[row["target"]]])
        attr_list.append(row["metaedge_encoded"])

edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
edge_attr = torch.tensor(attr_list, dtype=torch.long)
split = split_data_stratified(edge_index, edge_attr, nodes, val_ratio=0.2, test_ratio=0.2, seed=seed)
task_edges = {task: split for task in tasks}

train_edge_index = split['train']['edge_index']
degrees_train = degree(train_edge_index[0], num_nodes=num_nodes).cpu().numpy()

def create_node_features(nodes_df, num_kind_classes, edge_index_subset=None):
    kind_encoded = torch.tensor(nodes_df["kind_encoded"].values, dtype=torch.long)
    kind_onehot = F.one_hot(kind_encoded, num_classes=num_kind_classes).float()
    random_feats = torch.empty(num_nodes, 128, device=device)
    torch.nn.init.xavier_uniform_(random_feats)
    x = torch.cat([random_feats, kind_onehot.to(device)], dim=1)
    return x

node_features_train = create_node_features(nodes, num_kind_classes, split['train']['edge_index'])
node_features_val   = create_node_features(nodes, num_kind_classes, split['val']['edge_index'])
node_features_test  = create_node_features(nodes, num_kind_classes, split['test']['edge_index'])

# --------------------------- PyG Data Objects ---------------------------
train_data = Data(x=node_features_train, edge_index=split['train']['edge_index'], edge_attr=split['train']['edge_attr']).to(device)
val_data = Data(x=node_features_val, edge_index=split['val']['edge_index'], edge_attr=split['val']['edge_attr']).to(device)
test_data = Data(x=node_features_test, edge_index=split['test']['edge_index'], edge_attr=split['test']['edge_attr']).to(device)

# --------------------------- Full Existing Set ---------------------------
def create_existing_set(edge_index_subset):
    existing = set()
    for src, tgt in zip(edge_index_subset[0].cpu().numpy(), edge_index_subset[1].cpu().numpy()):
        existing.add((int(src), int(tgt)))
        existing.add((int(tgt), int(src)))
    return existing

train_existing_set = create_existing_set(split['train']['edge_index'])
val_existing_set   = create_existing_set(split['val']['edge_index'])
test_existing_set  = create_existing_set(split['test']['edge_index'])

# --------------------------- Utilities ---------------------------
def drop_edge(edge_index, edge_attr, drop_rate=0.1):
    num_edges = edge_index.size(1)
    if num_edges == 0:
        return edge_index, edge_attr
    keep_mask = torch.rand(num_edges, device=edge_index.device) > drop_rate
    return edge_index[:, keep_mask], edge_attr[keep_mask]

def sample_hard_negative_edges(model, embeddings, edge_index, num_neg_samples, num_nodes, task, existing_edges=None):

    if existing_edges is None:
        existing_edges = set()
        if edge_index is not None and edge_index.numel() > 0:
            src = edge_index[0].cpu().numpy()
            tgt = edge_index[1].cpu().numpy()
            for s, t in zip(src, tgt):
                existing_edges.add((int(s), int(t)))
                existing_edges.add((int(t), int(s)))
    if task == "DaG":
        candidate_factor = 0.1
    elif task == "CrC":
        candidate_factor = 0.1
    elif task == "CaD":
        candidate_factor = 1.0
    else:
        candidate_factor = 0.5

    candidate_neg_samples = max(1, int(num_neg_samples * candidate_factor))
    src_kind, dst_kind = task_to_kinds[task]
    src_nodes = [n for n in kind_to_nodes[src_kind] if 0 <= n < num_nodes]
    dst_nodes = [n for n in kind_to_nodes[dst_kind] if 0 <= n < num_nodes]

    if not src_nodes or not dst_nodes or num_neg_samples <= 0:
        return (torch.empty((2, 0), dtype=torch.long, device=device),
                torch.empty((0,), dtype=torch.long, device=device))

    src_deg = degrees_train[src_nodes] + 1e-6
    src_probs = src_deg / src_deg.sum()
    dst_deg = degrees_train[dst_nodes] + 1e-6
    dst_probs = dst_deg / dst_deg.sum()
    candidate_edges = []
    seen = set()
    attempts = 0
    max_attempts = candidate_neg_samples * 10

    while len(candidate_edges) < candidate_neg_samples and attempts < max_attempts:
        attempts += 1
        num_to_sample = min(candidate_neg_samples * 3, candidate_neg_samples - len(candidate_edges) + 20)
        src = np.random.choice(src_nodes, num_to_sample, p=src_probs)
        dst = np.random.choice(dst_nodes, num_to_sample, p=dst_probs)
        for s, d in zip(src, dst):
            if s == d:
                continue
            pair = (int(s), int(d))
            if pair in existing_edges or (d, s) in existing_edges:
                continue
            if pair in seen:
                continue
            candidate_edges.append([s, d])
            seen.add(pair)
            if len(candidate_edges) >= candidate_neg_samples:
                break
    if not candidate_edges:
        return (torch.empty((2, 0), dtype=torch.long, device=device),
                torch.empty((0,), dtype=torch.long, device=device))

    candidate_edges = torch.tensor(candidate_edges[:candidate_neg_samples], dtype=torch.long, device=device).t()
    with torch.no_grad():
        num_candidates = candidate_edges.size(1)
        task_id = le_metaedge.transform([task])[0]
        task_edge_attr = torch.full((num_candidates,), task_id, dtype=torch.long, device=device)
        task_edge_type_emb = model.edge_type_embeddings(task_edge_attr)

        scores = model.predict(embeddings, candidate_edges, task_edge_type_emb, task).flatten()
        k = min(num_neg_samples, num_candidates)
        if k > 0:
            _, top_indices = torch.topk(scores, k)
            hard_neg_edges = candidate_edges[:, top_indices]
            hard_neg_attr = task_edge_attr[top_indices]
        else:
            hard_neg_edges = torch.empty((2, 0), dtype=torch.long, device=device)
            hard_neg_attr = torch.empty((0,), dtype=torch.long, device=device)

    return hard_neg_edges, hard_neg_attr

def hidden_state_matching_loss_list(student_hiddens, teacher_hiddens, layer_weights=None):
    assert len(student_hiddens) == len(teacher_hiddens)
    if layer_weights is None:
        layer_weights = [1.0] * len(student_hiddens)
    loss = 0.0
    for w, sh, th in zip(layer_weights, student_hiddens, teacher_hiddens):
        loss = loss + w * F.mse_loss(sh, th, reduction='mean')
    return loss

def batch_sampling(edge_pairs, edge_attr, edge_type_emb, batch_size, device):
    num_edges = edge_pairs.size(1)
    if num_edges == 0:
        return edge_pairs, edge_attr, edge_type_emb
    indices = torch.randperm(num_edges)[:min(batch_size, num_edges)]
    return edge_pairs[:, indices].to(device), edge_attr[indices].to(device), edge_type_emb[indices].to(device)

def get_cosine_temperature(epoch, max_epochs, initial_temperature=2.0, min_temperature=1.0):
    progress = epoch / max_epochs
    temperature = min_temperature + (initial_temperature - min_temperature) * (1 + np.cos(np.pi * progress)) / 2
    return temperature

# --------------------------- Models ---------------------------
class MultiTaskGraphSAGE(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, task_outputs, num_edge_types):
        super(MultiTaskGraphSAGE, self).__init__()
        self.conv1 = SAGEConv(input_dim, hidden_dim, aggr='mean')
        self.conv2 = SAGEConv(hidden_dim, hidden_dim, aggr='mean')
        self.edge_type_embeddings = torch.nn.Embedding(num_edge_types, hidden_dim)
        self.task_heads = ModuleDict({task: torch.nn.Sequential(Linear(hidden_dim * 3, hidden_dim),
                                                                LeakyReLU(negative_slope=0.2), Linear(hidden_dim, output_dim))
                                      for task, output_dim in task_outputs.items()})
        self.dropout = torch.nn.Dropout(0.2)
        self.leaky_relu = LeakyReLU(negative_slope=0.2)
    def forward(self, x, edge_index, edge_attr, return_hidden=False):
        x = x.clone()
        h1 = self.leaky_relu(self.conv1(x, edge_index))
        h2 = self.leaky_relu(self.conv2(h1, edge_index))
        z = self.dropout(h2)
        edge_bias = self.edge_type_embeddings(edge_attr).mean(dim=0)
        z = z + edge_bias
        if return_hidden:
            return z, [h1, h2]
        return z

    def predict(self, embeddings, edge_pairs, edge_type_emb, task):
        if edge_pairs.size(1) == 0:
            return torch.tensor([], device=embeddings.device)
        assert edge_pairs.size(1) == edge_type_emb.size(0)
        src = embeddings[edge_pairs[0]]
        dst = embeddings[edge_pairs[1]]
        concat = torch.cat([src, dst, edge_type_emb], dim=1)
        logits = self.task_heads[task](concat)
        return logits

class TeacherModel(MultiTaskGraphSAGE):
    def __init__(self, input_dim, hidden_dim, task_outputs, num_edge_types):
        super(TeacherModel, self).__init__(input_dim, hidden_dim, task_outputs, num_edge_types)
        self.log_vars = torch.nn.Parameter(torch.zeros(len(task_outputs)))

class StudentModel(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, task_outputs, num_edge_types, teacher_hidden_dim):
        super(StudentModel, self).__init__()
        self.conv1 = SAGEConv(input_dim, hidden_dim, aggr='mean')
        self.edge_type_embeddings = torch.nn.Embedding(num_edge_types, hidden_dim)
        self.task_heads = ModuleDict({task: torch.nn.Sequential(Linear(hidden_dim * 3, hidden_dim), Linear(hidden_dim, output_dim))
                                      for task, output_dim in task_outputs.items()})
        self.log_vars = torch.nn.Parameter(torch.zeros(len(task_outputs)))
        self.proj1 = Linear(hidden_dim, teacher_hidden_dim)
    def forward(self, x, edge_index, edge_attr, return_hidden=False):
        x = x.clone()
        h1 = self.conv1(x, edge_index)
        z = h1
        edge_bias = self.edge_type_embeddings(edge_attr).mean(dim=0)
        z = z + edge_bias
        if return_hidden:
            ph1 = self.proj1(h1)
            return z, [ph1]
        return z

    def predict(self, embeddings, edge_pairs, edge_type_emb, task):
        if edge_pairs.size(1) == 0:
            return torch.tensor([], device=embeddings.device)
        assert edge_pairs.size(1) == edge_type_emb.size(0)
        src = embeddings[edge_pairs[0]]
        dst = embeddings[edge_pairs[1]]
        concat = torch.cat([src, dst, edge_type_emb], dim=1)
        logits = self.task_heads[task](concat)
        return logits

# --------------------------- Losses ---------------------------
class WeightedBCEWithLogitsLoss(torch.nn.Module):
    def __init__(self, pos_weight_dict=None, label_smoothing=0.1):
        super().__init__()
        self.pos_weight_dict = pos_weight_dict or {}
        self.label_smoothing = label_smoothing
        self.base_criterion = BCEWithLogitsLoss(reduction='mean')

    def forward(self, logits, targets, task=None):
        pos_weight = self.pos_weight_dict.get(task, 1.0)
        if isinstance(pos_weight, (int, float)):
            pos_weight = torch.tensor([pos_weight], device=logits.device)
        smoothed = targets * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        self.base_criterion.pos_weight = pos_weight
        return self.base_criterion(logits.view_as(smoothed), smoothed)

def distillation_loss(student_logits, teacher_logits, temperature=2.0):
    s = torch.stack([torch.zeros_like(student_logits), student_logits], dim=-1) / temperature
    t = torch.stack([torch.zeros_like(teacher_logits), teacher_logits], dim=-1) / temperature
    s_logprob = F.log_softmax(s, dim=-1)
    t_prob = F.softmax(t, dim=-1)
    return F.kl_div(s_logprob, t_prob, reduction='batchmean') * (temperature ** 2)

class WeightedLossCombiner(torch.nn.Module):
    def __init__(self, alpha, beta, gamma):
        super(WeightedLossCombiner, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
    def forward(self, task_loss, distill_loss, hs_loss):
        return self.alpha * task_loss + self.beta * distill_loss + self.gamma * hs_loss

def mean_ci(data, confidence=0.95):
    data = np.array(data)
    mean = np.mean(data)
    sem = st.sem(data)
    h = sem * st.t.ppf((1 + confidence) / 2., len(data)-1)
    return mean, h

def calculate_flops(model, data, edge_index, edge_attr, task_edges, batch_size, num_nodes, num_edge_types, is_student=False):
    flops = 0.0
    input_dim = data.x.shape[1]
    hidden_dim = model.conv1.out_channels if hasattr(model, 'conv1') else 0
    num_edges = edge_index.shape[1]
    flops += 2 * num_edges * (input_dim * hidden_dim)
    flops += 2 * num_edges * hidden_dim

    if hasattr(model, 'conv2'):
        flops += 2 * num_edges * (hidden_dim * hidden_dim)
        flops += 2 * num_edges * hidden_dim

    if hasattr(model, 'edge_type_embeddings') and edge_attr is not None:
        emb_dim = model.edge_type_embeddings.embedding_dim
        flops += num_edges * emb_dim * 2
        flops += num_edges * emb_dim

    for task in task_edges:
        if task not in getattr(model, 'task_heads', {}):
            continue
        flops += batch_size * (hidden_dim * 3 * hidden_dim)
        flops += batch_size * hidden_dim
        flops += batch_size * (hidden_dim * 1)
        flops += batch_size * 1

    flops += num_nodes * hidden_dim * 6

    if is_student and hasattr(model, 'proj1'):
        proj_out = model.proj1.out_features
        flops += num_nodes * (hidden_dim * proj_out)
        flops += num_nodes * proj_out

    gflops = flops / 1e9
    return gflops

def get_memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    return mem_info.rss / 1024 / 1024

# --------------------------- Training / Eval ---------------------------
def train_teacher(model, train_data, task_edges, optimizer, criterion, device, batch_size, iterations=50):
    model.train()
    start_time = time.time()
    total_loss = 0.0
    num_nodes = train_data.x.shape[0]
    gflops = calculate_flops(model, train_data, train_data.edge_index, train_data.edge_attr, task_edges, batch_size, num_nodes, num_edge_types, is_student=False)
    mem_before = get_memory_usage()

    for _ in range(iterations):
        optimizer.zero_grad()
        edge_index, edge_attr = drop_edge(train_data.edge_index, train_data.edge_attr, drop_rate=0.1)
        embeddings = model(train_data.x, edge_index, edge_attr)
        task_losses = {task: torch.zeros(1, device=device, requires_grad=True) for task in tasks}

        for task, split_dict in task_edges.items():
            pos_pairs = split_dict['train']['edge_index'].to(device)
            pos_attr = split_dict['train']['edge_attr'].to(device)
            if pos_pairs.size(1) == 0:
                continue
            pos_type_emb = model.edge_type_embeddings(pos_attr)

            for _start in range(0, pos_pairs.size(1), batch_size):
                b_pos_pairs, b_pos_attr, b_pos_type_emb = batch_sampling(pos_pairs, pos_attr, pos_type_emb, batch_size, device)
                b_neg_pairs, b_neg_attr = sample_hard_negative_edges(model, embeddings, train_data.edge_index, b_pos_pairs.size(1), num_nodes, task, existing_edges=train_existing_set)
                b_neg_pairs = b_neg_pairs.to(device)
                b_neg_attr = b_neg_attr.to(device)
                b_neg_type_emb = model.edge_type_embeddings(b_neg_attr)
                pos_logits = model.predict(embeddings, b_pos_pairs, b_pos_type_emb, task)
                neg_logits = model.predict(embeddings, b_neg_pairs, b_neg_type_emb, task)
                all_logits = torch.cat([pos_logits, neg_logits], dim=0)
                all_labels = torch.cat([torch.ones_like(pos_logits), torch.zeros_like(neg_logits)], dim=0)
                loss = criterion(all_logits, all_labels, task=task)
                task_losses[task] = task_losses[task] + loss
        task_loss = torch.zeros(1, device=device, requires_grad=True)
        for i, task in enumerate(tasks):
            if task_losses[task].item() > 0:
                weighted = torch.exp(-model.log_vars[i]) * task_losses[task] + 0.5 * model.log_vars[i]
                task_loss = task_loss + weighted
        task_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += task_loss.item()
    mem_after = get_memory_usage()
    return (total_loss / iterations, time.time() - start_time, gflops, mem_after - mem_before)

def train_student(model, teacher_model, train_data, task_edges, optimizer, criterion, distillation_criterion, device, loss_combiner,
                  batch_size, iterations=50, epoch=0, max_epochs=50, layer_weights=None):

    model.train()
    teacher_model.eval()
    start_time = time.time()
    total_loss = 0.0
    total_distill_loss = 0.0
    total_hs_loss = 0.0
    num_nodes = train_data.x.shape[0]
    temperature = get_cosine_temperature(epoch, max_epochs)
    gflops = calculate_flops(model, train_data, train_data.edge_index, train_data.edge_attr, task_edges, batch_size, num_nodes, num_edge_types, is_student=True)
    mem_before = get_memory_usage()

    for _ in range(iterations):
        optimizer.zero_grad()
        edge_index, edge_attr = drop_edge(train_data.edge_index, train_data.edge_attr, drop_rate=0.1)
        with torch.no_grad():
            t_embed, t_hs = teacher_model(train_data.x, edge_index, edge_attr, return_hidden=True)
            t_hs = [t_hs[-1]]
        s_embed, s_hs_proj = model(train_data.x, edge_index, edge_attr, return_hidden=True)
        hs_loss = hidden_state_matching_loss_list(s_hs_proj, t_hs, layer_weights=layer_weights)
        task_losses = {task: torch.zeros(1, device=device, requires_grad=True) for task in tasks}
        distill_losses = {task: torch.zeros(1, device=device, requires_grad=True) for task in tasks}
        for task, split_dict in task_edges.items():
            pos_pairs = split_dict['train']['edge_index'].to(device)
            pos_attr = split_dict['train']['edge_attr'].to(device)
            if pos_pairs.size(1) == 0:
                continue
            for _start in range(0, pos_pairs.size(1), batch_size):
                s_pos_type_emb_full = model.edge_type_embeddings(pos_attr)
                b_pos_pairs, b_pos_attr, b_s_pos_type_emb = batch_sampling(pos_pairs, pos_attr, s_pos_type_emb_full, batch_size, device)
                b_neg_pairs, b_neg_attr = sample_hard_negative_edges(model, s_embed, train_data.edge_index, b_pos_pairs.size(1),
                                                                     num_nodes, task, existing_edges=train_existing_set)

                b_neg_pairs = b_neg_pairs.to(device)
                b_neg_attr = b_neg_attr.to(device)
                b_s_neg_type_emb = model.edge_type_embeddings(b_neg_attr)
                b_t_pos_type_emb = teacher_model.edge_type_embeddings(b_pos_attr)
                b_t_neg_type_emb = teacher_model.edge_type_embeddings(b_neg_attr)
                s_pos_logits = model.predict(s_embed, b_pos_pairs, b_s_pos_type_emb, task)
                s_neg_logits = model.predict(s_embed, b_neg_pairs, b_s_neg_type_emb, task)
                s_all_logits = torch.cat([s_pos_logits, s_neg_logits], dim=0)
                all_labels = torch.cat([torch.ones_like(s_pos_logits), torch.zeros_like(s_neg_logits)], dim=0)
                loss_task = criterion(s_all_logits, all_labels, task=task)

                with torch.no_grad():
                    t_pos_logits = teacher_model.predict(t_embed, b_pos_pairs, b_t_pos_type_emb, task)
                    t_neg_logits = teacher_model.predict(t_embed, b_neg_pairs, b_t_neg_type_emb, task)
                    t_all_logits = torch.cat([t_pos_logits, t_neg_logits], dim=0)
                loss_distill = distillation_criterion(s_all_logits, t_all_logits, temperature=temperature)
                task_losses[task] = task_losses[task] + loss_task
                distill_losses[task] = distill_losses[task] + loss_distill
                total_distill_loss += loss_distill.item()
        total_hs_loss += hs_loss.item()
        loss_task_sum = torch.zeros(1, device=device, requires_grad=True)
        distill_sum = torch.zeros(1, device=device, requires_grad=True)

        for i, task in enumerate(tasks):
            if task_losses[task].item() > 0:
                weighted_hard = torch.exp(-model.log_vars[i]) * task_losses[task] + 0.5 * model.log_vars[i]
                weighted_distill = torch.exp(-model.log_vars[i]) * distill_losses[task] + 0.5 * model.log_vars[i]
                loss_task_sum = loss_task_sum + weighted_hard
                distill_sum = distill_sum + weighted_distill
        combined_loss = loss_combiner(loss_task_sum, distill_sum, hs_loss)
        combined_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += combined_loss.item()
    mem_after = get_memory_usage()
    return (total_loss / iterations, time.time() - start_time, total_hs_loss / iterations, gflops, mem_after - mem_before)

def test_model(model, test_data, task_edges, split='test', iterations=50):
    model.eval()
    start = time.time()
    all_metrics = {task: {'auc': [], 'aupr': [], 'accuracy': [], 'f1': [], 'precision': [], 'threshold': []} for task in task_edges}
    num_nodes = test_data.x.shape[0]

    for _ in range(iterations):
        with torch.no_grad():
            emb = model(test_data.x, test_data.edge_index, test_data.edge_attr)
            for task, edges in task_edges.items():
                edge_split = edges[split]['edge_index']
                edge_attr = edges[split]['edge_attr']
                if edge_split.size(1) == 0:
                    for k in all_metrics[task]:
                        all_metrics[task][k].append(0.0 if k != 'threshold' else 0.5)
                    continue
                type_emb_pos = model.edge_type_embeddings(edge_attr)
                pos_logits = model.predict(emb, edge_split, type_emb_pos, task)
                neg_edges, neg_attr = sample_hard_negative_edges(model, emb, test_data.edge_index, edge_split.size(1), num_nodes, task,
                                                                 existing_edges=test_existing_set)
                type_emb_neg = model.edge_type_embeddings(neg_attr)
                neg_logits = model.predict(emb, neg_edges, type_emb_neg, task)
                logits = torch.cat([pos_logits, neg_logits]).cpu().numpy()
                labels = np.concatenate([np.ones_like(pos_logits.cpu().numpy()), np.zeros_like(neg_logits.cpu().numpy())])
                precision, recall, thresholds = precision_recall_curve(labels, logits)
                f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
                optimal_threshold = thresholds[np.argmax(f1_scores)] if len(thresholds) > 0 else 0.0
                binary_preds = (logits >= optimal_threshold).astype(int)
                auc = roc_auc_score(labels, logits) if len(np.unique(labels)) > 1 else 0.0
                aupr = average_precision_score(labels, logits) if len(np.unique(labels)) > 1 else 0.0
                acc = accuracy_score(labels, binary_preds)
                f1 = f1_score(labels, binary_preds, zero_division=0)
                prec = precision_score(labels, binary_preds, zero_division=0)
                all_metrics[task]['auc'].append(auc)
                all_metrics[task]['aupr'].append(aupr)
                all_metrics[task]['accuracy'].append(acc)
                all_metrics[task]['f1'].append(f1)
                all_metrics[task]['precision'].append(prec)
                all_metrics[task]['threshold'].append(optimal_threshold)
    final_metrics = {}
    for task in tasks:
        metrics = {}
        for k in ['auc', 'aupr', 'accuracy', 'f1', 'precision']:
            vals = all_metrics[task][k]
            mean, h = mean_ci(vals)
            metrics[k] = {'mean': mean, 'ci': h}
        final_metrics[task] = metrics
    final_metrics['inference_time'] = {'avg': time.time() - start}
    return final_metrics

def validate_model(model, val_data, task_edges, split='val', iterations=50):
    return test_model(model, val_data, task_edges, split, iterations)

def print_metrics(prefix, metrics):
    for task in tasks:
        if task in metrics:
            print(f"{prefix} {task}: ", end='')
            for k, v in metrics[task].items():
                print(f"{k.upper()}: {v['mean']:.4f} ± {v['ci']:.4f} ", end='')
            print()

# --------------------------- Hyperparams & Train ---------------------------

hidden_dim_t = 128
hidden_dim_s = 64
task_outputs = {task: 1 for task in tasks}
num_edge_types = len(le_metaedge.classes_)
batch_size = 256
epochs_t = 100
epochs_s = 50

criterion = WeightedBCEWithLogitsLoss(pos_weight_dict=task_pos_weights, label_smoothing=0.1)
teacher_model = TeacherModel(input_dim=node_features_train.shape[1], hidden_dim=hidden_dim_t, task_outputs=task_outputs, num_edge_types=num_edge_types).to(device)
optimizer_teacher = torch.optim.Adam(teacher_model.parameters(), lr=0.0008, weight_decay=0.0005)

teacher_gflops_list = []
teacher_mem_list = []

print("Training Teacher Model...")
for epoch in range(epochs_t):
    loss_t, t_time, gflops_t, mem_usage_t = train_teacher(teacher_model, train_data, task_edges, optimizer_teacher, criterion, device, batch_size)
    teacher_gflops_list.append(gflops_t)
    teacher_mem_list.append(mem_usage_t)
    val_metrics = validate_model(teacher_model, val_data, task_edges)
    print(f"Epoch {epoch + 1}/{epochs_t} - Loss: {loss_t:.4f}, Time: {t_time:.4f}s, GFLOPs: {gflops_t:.4f}, Memory Usage (MB): {mem_usage_t:.2f}")
    print_metrics("Val Teacher", val_metrics)

alpha, beta, gamma = 0.5, 0.3, 0.2
student_model = StudentModel(input_dim=node_features_train.shape[1], hidden_dim=hidden_dim_s, task_outputs=task_outputs,
                             num_edge_types=num_edge_types, teacher_hidden_dim=hidden_dim_t).to(device)

optimizer_student = torch.optim.Adam(student_model.parameters(), lr=0.0008, weight_decay=0.0005)
loss_combiner = WeightedLossCombiner(alpha=alpha, beta=beta, gamma=gamma).to(device)

student_gflops_list = []
student_mem_list = []

print(f"\nTraining Student Model (alpha={alpha:.2f}, beta={beta:.2f}, gamma={gamma:.2f})...")
for epoch in range(epochs_s):
    loss_s, s_time, hs_l, gflops_s, mem_usage_s = train_student(student_model, teacher_model, train_data, task_edges, optimizer_student, criterion, distillation_loss, device, loss_combiner, batch_size, epoch=epoch, max_epochs=epochs_s, layer_weights=[1.0])
    student_gflops_list.append(gflops_s)
    student_mem_list.append(mem_usage_s)
    val_metrics = validate_model(student_model, val_data, task_edges)
    print(f"Epoch {epoch + 1}/{epochs_s} - Loss: {loss_s:.4f}, Time: {s_time:.4f}s, HS Loss: {hs_l:.4f}, GFLOPs: {gflops_s:.4f}, Memory Usage (MB): {mem_usage_s:.2f}")
    print_metrics("Val Student", val_metrics)

test_metrics_student = test_model(student_model, test_data, task_edges, 'test')
test_metrics_teacher = test_model(teacher_model, test_data, task_edges, 'test')

print("\nStudent Model Test Results:")
print_metrics("Test Student", test_metrics_student)
print(f"Inference Time: {test_metrics_student['inference_time']['avg']:.4f}s")

print("\nTeacher Model Test Results:")
print_metrics("Test Teacher", test_metrics_teacher)
print(f"Inference Time: {test_metrics_teacher['inference_time']['avg']:.4f}s")

avg_teacher_gflops = np.mean(teacher_gflops_list)
avg_teacher_mem = np.mean(teacher_mem_list)
avg_student_gflops = np.mean(student_gflops_list)
avg_student_mem = np.mean(student_mem_list)

print("\nComparative Averages:")
print(f"Average GFLOPs - Teacher: {avg_teacher_gflops:.4f}, Student: {avg_student_gflops:.4f}")
print(f"Average Memory Usage (MB) - Teacher: {avg_teacher_mem:.2f}, Student: {avg_student_mem:.2f}")